In [1]:
import pandas as pd
import pyomo.environ as pyo
from pyomo.opt import SolverFactory
from pathlib import Path
import numpy as np
import math
from collections import defaultdict
from itertools import combinations

In [2]:
programme_courses = pd.read_csv("Cleaned_Data/Cleaned Programme-Courses.csv")
rooms = pd.read_csv("Cleaned_Data/Cleaned Rooms and Room Types .csv")
EMR = pd.read_csv("Cleaned_Data/Cleaned Event Module Room.csv")
EMR["Event Size"] = pd.to_numeric(EMR["Event Size"], errors="coerce").fillna(0)
SPME = pd.read_csv("2024-5 Student Programme Module Event.csv")

In [3]:
rooms_toy = rooms[rooms["Campus"] == "Kings Buildings"]
rooms_toy = rooms_toy[["Id", "Capacity", "Building.1", "Specialist room type"]]
#rooms_toy.loc[len(rooms_toy)] = ["0787_00_XX", 1000, "Sheep Shed"]
print(rooms_toy)
print("rows:", len(rooms_toy))

                  Id  Capacity             Building.1 Specialist room type
439  0616_00_00.01-E      86.0                 Alrick     General Teaching
440  0616_01_01.03-E      80.0                 Alrick  Computer Laboratory
441  0616_02_02.02-E      72.0                 Alrick     General Teaching
442   0640_00_G.0070     313.0          Ashworth Labs     General Teaching
443   0640_00_G.0080     116.0          Ashworth Labs           Laboratory
..               ...       ...                    ...                  ...
677    0621_01_1.304       NaN                   Crew         Meeting Room
678     0648_00_G.02       NaN    Roger Land Building         Meeting Room
679     0648_00_G.13       NaN    Roger Land Building         Meeting Room
680     0648_01_1.55       NaN    Roger Land Building         Meeting Room
681     0670_01_1.07       NaN  Waddington Building 1         Meeting Room

[121 rows x 4 columns]
rows: 121


In [4]:
target_buildings = [
    "JCMB",
]

rooms_toy = rooms[
    (rooms["Campus"] == "Kings Buildings") &
    (rooms["Building.1"].isin(target_buildings))
].copy()


rooms_toy["Specialist room type"] = rooms_toy["Specialist room type"].replace(
    "Teaching Studio",
    "General Teaching"
)


rooms_toy = rooms_toy[["Id", "Capacity", "Building.1", "Specialist room type"]]

print(rooms_toy)
print("rows:", len(rooms_toy))


                  Id  Capacity Building.1 Specialist room type
473   0613_01_1.1104     104.0       JCMB           Laboratory
474  0613_01_1.1206C      72.0       JCMB     General Teaching
475   0613_01_1.1208      66.0       JCMB  Computer Laboratory
476   0613_01_1.1501      62.0       JCMB     General Teaching
477   0613_01_1.1512      30.0       JCMB  Computer Laboratory
478   0613_02_2.2205      12.0       JCMB           Laboratory
479   0613_02_2.2206       7.0       JCMB           Laboratory
480   0613_02_2.2209      60.0       JCMB           Laboratory
481   0613_02_2.2901      14.0       JCMB     General Teaching
482   0613_02_2.3902     197.0       JCMB     General Teaching
483   0613_02_2.3905     150.0       JCMB     General Teaching
484   0613_02_2.3908     104.0       JCMB     General Teaching
485   0613_03_3.3201      18.0       JCMB           Laboratory
486   0613_03_3.3206      21.0       JCMB           Laboratory
487   0613_03_3.3208      32.0       JCMB           Lab

In [5]:
print(rooms_toy["Specialist room type"].value_counts())

Specialist room type
General Teaching       18
Laboratory             15
Computer Laboratory     3
Name: count, dtype: int64


In [6]:
def keep_chosen_weeks(weeks_value):
    if pd.isna(weeks_value):
        return None

    weeks = []
    for part in str(weeks_value).split(","):
        part = part.strip()
        if "-" in part:
            start, end = map(int, part.split("-"))
            weeks.extend(range(start, end + 1))
        else:
            weeks.append(int(part))

    kept = sorted(set(w for w in weeks if 26 <= w <= 36))

    if not kept:
        return None

    return ",".join(map(str, kept))


EMR_toy = EMR[EMR["Module Department"].isin(["School of Mathematics"])].copy()

EMR_toy = EMR_toy[EMR_toy["Campus"] != "BioQuarter"].copy()

EMR_toy["Weeks"] = EMR_toy["Weeks"].apply(keep_chosen_weeks)
EMR_toy = EMR_toy[EMR_toy["Weeks"].notna()]

EMR_toy = EMR_toy[[
    "Module Department",
    "Building",
    "Event ID",
    "Module Code",
    "Duration (minutes)",
    "Event Size",
    "WholeClass",
    "Weeks",
    "Room type 2",
    "Online Delivery"
]]

EMR_toy["Room type 2"] = EMR_toy["Room type 2"].replace(
    "Centrally Allocated Space",
    "General Teaching"
)

EMR_toy["Room type 2"] = EMR_toy["Room type 2"].replace(
    "Foyer",
    "General Teaching"
)

EMR_toy["Room type 2"] = EMR_toy["Room type 2"].replace(
    "Teaching Studio",
    "General Teaching"
)

EMR_toy["Duration (minutes)"] = EMR_toy["Duration (minutes)"].clip(upper=480)

print(EMR_toy.head())
print("rows:", len(EMR_toy))
print(EMR_toy.isna().sum())


          Module Department Building            Event ID  \
1335  School of Mathematics     JCMB        E:N98GJ6OX7O   
2157  School of Mathematics  Nucleus  E:IONUAIAG2L-00002   
2158  School of Mathematics  Nucleus  E:ECHY65JQZW-00002   
2160  School of Mathematics  Nucleus        E:IONUAIAG2L   
2166  School of Mathematics  Nucleus  E:IONUAIAG2L-00001   

                    Module Code  Duration (minutes)  Event Size  WholeClass  \
1335    MATH10102_SS1_YR_2024/5                 110        80.0        True   
2157  MATH08065_SV1_SEM2_2024/5                 110        15.0       False   
2158  MATH08065_SV1_SEM2_2024/5                 110        15.0       False   
2160  MATH08065_SV1_SEM2_2024/5                 110        15.0       False   
2166  MATH08065_SV1_SEM2_2024/5                 110        15.0       False   

                              Weeks       Room type 2 Online Delivery  
1335                          26,27  General Teaching             NaN  
2157  26,27,28,29,30

In [7]:
print(EMR_toy["Room type 2"].value_counts())

Room type 2
General Teaching       570
Computer Laboratory     45
Name: count, dtype: int64


changed teaching studio to general teaching as they are basically the same thing

In [8]:
event_programme_df = SPME[["Event ID", "Programme Code-Year"]]

event_programmes = (
    event_programme_df
    .drop_duplicates()
    .groupby("Event ID")["Programme Code-Year"]
    .agg(list)
    .to_dict()
)

event_programme_toy = (
    event_programme_df
    .drop_duplicates()
    .groupby("Event ID", as_index=False)["Programme Code-Year"]
    .agg(list)
)
display(event_programme_toy.head())
print(event_programme_toy.shape)
print(event_programme_toy.isna().sum())

,Event ID,Programme Code-Year
0,E:114K59WCRK,"[UTELEEB_YR4_2024/5, UTEMENM_YR5_2024/5, UTELE..."
1,E:117T8ZQ92F,"[UTCHENB_YR4_2024/5, UTBENCHEET1F_YR4_2024/5, ..."
2,E:117Z6VV1GW,[PTMARTCOAP1F_YR1_2024/5]
3,E:11FIIGMRZ7,"[UTARCSA_YR2_2024/5, UTARCHA_YR2_2024/5, UTMAH..."
4,E:11FIIGMRZ7-00001,"[UTARCSA_YR2_2024/5, UTARCHA_YR2_2024/5, UTMAH..."


(29105, 2)
Event ID               0
Programme Code-Year    0
dtype: int64


In [9]:
df_events = EMR_toy.merge(
    event_programme_toy,
    on="Event ID",
    how="left"
)
print(df_events.head())
print("rows:", len(df_events))
df_events.isna().sum()

       Module Department Building            Event ID  \
0  School of Mathematics     JCMB        E:N98GJ6OX7O   
1  School of Mathematics  Nucleus  E:IONUAIAG2L-00002   
2  School of Mathematics  Nucleus  E:ECHY65JQZW-00002   
3  School of Mathematics  Nucleus        E:IONUAIAG2L   
4  School of Mathematics  Nucleus  E:IONUAIAG2L-00001   

                 Module Code  Duration (minutes)  Event Size  WholeClass  \
0    MATH10102_SS1_YR_2024/5                 110        80.0        True   
1  MATH08065_SV1_SEM2_2024/5                 110        15.0       False   
2  MATH08065_SV1_SEM2_2024/5                 110        15.0       False   
3  MATH08065_SV1_SEM2_2024/5                 110        15.0       False   
4  MATH08065_SV1_SEM2_2024/5                 110        15.0       False   

                           Weeks       Room type 2 Online Delivery  \
0                          26,27  General Teaching             NaN   
1  26,27,28,29,30,32,33,34,35,36  General Teaching          

Module Department        0
Building                 0
Event ID                 0
Module Code              0
Duration (minutes)       0
Event Size               0
WholeClass               0
Weeks                    0
Room type 2              0
Online Delivery        615
Programme Code-Year     13
dtype: int64

In [10]:
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
weeks = range(26, 37)    # weeks 26 to 36 inclusive

# Mon-Thu: 09:00-17:00 = 8 one-hour slots
# Friday: 09:00-12:00 = 3 one-hour slots
slots = range(1, 9)   # global slot index
slots_by_day = {
    "Monday": range(1, 9),
    "Tuesday": range(1, 9),
    "Wednesday": range(1, 9),
    "Thursday": range(1, 9),
    "Friday": range(1, 4),
}

timeslots = pd.DataFrame(
    [(w, d, p) for w in weeks for d in days for p in slots_by_day[d]],
    columns=["Week", "Day", "Period"]
)

print(timeslots.head())
print("Total timeslots:", len(timeslots))

   Week     Day  Period
0    26  Monday       1
1    26  Monday       2
2    26  Monday       3
3    26  Monday       4
4    26  Monday       5
Total timeslots: 385


### Setup

In [11]:
slot_minutes = 60 # duration of each slot

# map: Event -> duration / room type / weeks of occurence
event_to_duration = df_events.set_index("Event ID")["Duration (minutes)"].to_dict()
event_to_roomtype = df_events.set_index("Event ID")["Room type 2"].to_dict()
event_to_weeks = (
    df_events.groupby("Event ID")["Weeks"]
    .apply(lambda s: sorted(set(int(w.strip()) for v in s.dropna() for w in str(v).split(","))))
    .to_dict()
)

In [12]:
tmp = df_events[["Event ID", "Programme Code-Year", "WholeClass"]].copy()

# change list of programmes to one row per (event, programme)
tmp["Programme Code-Year"] = tmp["Programme Code-Year"].apply(
    lambda x: x if isinstance(x, list) else [x]
)
tmp = tmp.explode("Programme Code-Year").dropna(subset=["Programme Code-Year"])

# normalize WholeClass to binary
tmp["WholeClass_bin"] = (
    tmp["WholeClass"]
    .astype(str).str.strip().str.upper()
    .map({"TRUE": 1, "FALSE": 0, "1": 1, "0": 0})
    .fillna(0)
    .astype(int)
)

whole_ep = tmp.groupby(["Event ID", "Programme Code-Year"])["WholeClass_bin"].max()

EP = list(whole_ep.index)
WholeEP = whole_ep.to_dict()
Pset = sorted(tmp["Programme Code-Year"].unique())

print("Unique events:", df_events["Event ID"].nunique())
print("Unique (event,programme) pairs:", len(EP))


Unique events: 615
Unique (event,programme) pairs: 4111


In [13]:
## feasible start times for events

# Duration of events in slots
L = {e: int(math.ceil(event_to_duration[e] / slot_minutes)) for e in event_to_duration.keys()}

E = sorted(event_to_duration.keys())

# Room availability by type
Avail = rooms_toy.groupby("Specialist room type")["Id"].count().to_dict()
ROOMTYPES = sorted(set(event_to_roomtype.values()))

last_slot_by_day = {d: max(slots_by_day[d]) for d in days}

# Feasible start slots per event, by day
feasible_starts = {
    e: {
        d: [s for s in slots_by_day[d] if s <= last_slot_by_day[d] - L[e] + 1]
        for d in days
    }
    for e in E
}

In [14]:
tmp_weeks = df_events[["Event ID", "Weeks"]].copy()

tmp_weeks["Weeks"] = tmp_weeks["Weeks"].astype(str).str.split(",")
tmp_weeks = tmp_weeks.explode("Weeks")
tmp_weeks["Weeks"] = tmp_weeks["Weeks"].str.strip()

events_by_week = (
    tmp_weeks.groupby("Weeks")["Event ID"]
    .nunique()
    .reset_index(name="Number of Events")
    .sort_values("Weeks", key=lambda s: s.astype(int))
)

#display(events_by_week)
print(events_by_week)


  Weeks  Number of Events
0    26               382
1    27               447
2    28               423
3    29               451
4    30               408
5    32               437
6    33               409
7    34               452
8    35               410
9    36               429


### Model

In [15]:
m1 = pyo.ConcreteModel()

m1.E = pyo.Set(initialize=E) # all events
m1.W = pyo.Set(initialize=weeks) # all weeks (1..52)
m1.D = pyo.Set(initialize=days) # all days
m1.S = pyo.Set(initialize=slots) # all slots 


m1.L = pyo.Param(m1.E, initialize=L, within=pyo.PositiveIntegers) # duration of events (given in slots)
m1.RoomType = pyo.Param(m1.E, initialize=event_to_roomtype, within=pyo.Any) # room type requirement per event

m1.P = pyo.Set(initialize=Pset) # all programmes (from Programme Code-Year), needed for WholeClass constraints
m1.EP = pyo.Set(dimen=2, initialize=EP) # all (event,programme) pairs
m1.WholeEP = pyo.Param(m1.EP, initialize=WholeEP, within=pyo.Binary, default=0) # 1 if that (event,programme) pair is WholeClass, else 0

In [16]:
# Build based on feasible start slots and feasible weeks
X_index = [
    (e, w, d, s)
    for e in E
    for w in event_to_weeks[e]
    for d in days
    for s in feasible_starts[e][d]
]

m1.X = pyo.Set(dimen=4, initialize=X_index)
m1.x = pyo.Var(m1.X, domain=pyo.Binary) # 1 if event e starts in week w on day d at slot s

### Constraints

In [17]:
# Same day and Start slot across all of an event's weeks. 

# = 1 if event e starts on day d and timeslot s
m1.Y = pyo.Set(dimen=3,initialize=[(e,d,s) for e in E for d in days for s in feasible_starts[e][d]])
m1.y = pyo.Var(m1.Y, domain=pyo.Binary)

# each event must choose exactly one (day,startslot) pair
def one_pattern(m, e):
    return sum(m.y[e,d,s] for d in m.D for s in feasible_starts[e][d]) == 1
m1.one_pattern = pyo.Constraint(m1.E, rule=one_pattern)

# each event must have same day and startslot in each week
def link_pattern(m, e, w, d, s):
    if (e,d,s) not in m.Y:
        return pyo.Constraint.Skip
    return m.x[e,w,d,s] == m.y[e,d,s]
m1.link_pattern = pyo.Constraint(m1.X, rule=link_pattern)


In [18]:
# 2) Room capacity constraints directly from start variables x
### allows general teaching to take place in any room
### allows teaching studio to take place in general teaching

NO_ROOM = "No room required"
Avail = rooms_toy.groupby("Specialist room type")["Id"].count().to_dict()
ALL_ROOM_TYPES = tuple(sorted(Avail))

def compatible_room_types(required_type):
    if pd.isna(required_type) or required_type == NO_ROOM:
        return tuple()
    if required_type == "General Teaching":
        return ALL_ROOM_TYPES
    if required_type == "Teaching Studio":
        return tuple(rt for rt in ["General Teaching", "Teaching Studio"] if rt in Avail)
    return (required_type,) if required_type in Avail else tuple()

event_room_options = {e: compatible_room_types(event_to_roomtype[e]) for e in E}

# Use pool constraints: events whose allowed rooms are contained in a pool
# cannot exceed the total capacity of that pool.
room_pools = sorted({opts for opts in event_room_options.values() if opts}, key=lambda x: (len(x), x))
pool_labels = {pool: " | ".join(pool) for pool in room_pools}
pool_caps = {pool_labels[pool]: sum(Avail[rt] for rt in pool) for pool in room_pools}

m1.R = pyo.Set(initialize=list(pool_caps.keys()))

starts_covering_wdst = defaultdict(list)

for e in E:
    opts = set(event_room_options[e])
    if not opts:
        continue

    Le = L[e]

    for w in event_to_weeks[e]:
        for d in days:
            for s in feasible_starts[e][d]:
                end_s = s + Le - 1
                for t in range(s, min(end_s, max(slots_by_day[d])) + 1):
                    for pool in room_pools:
                        if opts.issubset(pool):
                            key = (w, d, t, pool_labels[pool])
                            starts_covering_wdst[key].append((e, w, d, s))

RoomCap_index = list(starts_covering_wdst.keys())
m1.RoomCapIndex = pyo.Set(dimen=4, initialize=RoomCap_index)

def room_cap_rule(m, w, d, t, r):
    idxs = starts_covering_wdst[(w, d, t, r)]
    return sum(m.x[e, w, d, s] for (e, w, d, s) in idxs) <= pool_caps[r]

m1.room_cap = pyo.Constraint(m1.RoomCapIndex, rule=room_cap_rule)

### Solve

In [19]:
m1.obj = pyo.Objective(expr=0, sense=pyo.minimize)

In [20]:
solver = pyo.SolverFactory("gurobi")

# Optional but useful
#solver.options["MIPGap"] = 0.0
#solver.options["TimeLimit"] = 600  

res = solver.solve(m1, tee=True)
print("Termination:", res.solver.termination_condition)
print("Status:", res.solver.status)


Read LP format model from file C:\Users\asher\AppData\Local\Temp\tmp42q4ob95.pyomo.lp
Reading time = 1.98 seconds
x1: 140945 rows, 160026 columns, 499370 nonzeros
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i5-10210U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 140945 rows, 160026 columns and 499370 nonzeros (Min)
Model fingerprint: 0xca72758e
Model has 0 linear objective coefficients
Variable types: 1 continuous, 160025 integer (160025 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+01]
Found heuristic solution: objective 0.0000000

Explored 0 nodes (0 simplex iterations) in 0.17 seconds (0.03 work units)
Thread count was 1 (of 8 available processors)

Solution count 1: 0 

Optimal solution found (tolera

In [21]:
x_start = {idx: int(round(pyo.value(m1.x[idx]))) for idx in m1.X}
y_start = {idx: int(round(pyo.value(m1.y[idx]))) for idx in m1.Y}


In [22]:
# Whole-class programmes by event
whole_progs = {}
for (e, p), v in WholeEP.items():
    if v == 1:
        whole_progs.setdefault(e, set()).add(p)

# Build pair-specific conflict terms directly in the objective.
# This avoids introducing millions of overlap-link constraints.
pair_conflict_terms = {}
pair_weight = {}

for e1, e2 in combinations(sorted(E), 2):
    shared_progs = whole_progs.get(e1, set()) & whole_progs.get(e2, set())
    if not shared_progs:
        continue

    common_weeks = set(event_to_weeks[e1]) & set(event_to_weeks[e2])
    if not common_weeks:
        continue

    # Weight choice:
    # 1 -> penalize each clashing event pair once
    # len(shared_progs) -> penalize by number of shared whole-class programmes
    # len(shared_progs) * len(common_weeks) -> penalize by programmes and weeks affected
    pair_weight[(e1, e2)] = len(shared_progs) * len(common_weeks)

    terms = []
    L1, L2 = L[e1], L[e2]
    for d in days:
        for s1 in feasible_starts[e1][d]:
            end1 = s1 + L1 - 1
            for s2 in feasible_starts[e2][d]:
                end2 = s2 + L2 - 1
                if max(s1, s2) <= min(end1, end2):
                    terms.append((d, s1, s2))

    if terms:
        pair_conflict_terms[(e1, e2)] = terms

overlap_penalty = sum(
    pair_weight[e1, e2] * sum(
        m1.y[e1, d, s1] * m1.y[e2, d, s2]
        for (d, s1, s2) in pair_conflict_terms[e1, e2]
    )
    for (e1, e2) in pair_conflict_terms
)


In [23]:
# Replace the feasibility objective with the soft-constraint objective
m1.del_component(m1.obj)

w_o = 1
total_penalty = w_o * overlap_penalty
m1.obj = pyo.Objective(expr=total_penalty, sense=pyo.minimize)

# Load the stage-1 solution back as a MIP start
for idx, val in x_start.items():
    m1.x[idx].value = val

for idx, val in y_start.items():
    m1.y[idx].value = val

In [24]:
solver = pyo.SolverFactory("gurobi")
solver.options["NonConvex"] = 2
solver.options["MIPFocus"] = 1
#solver.options["Heuristics"] = 0.4
#solver.options["NoRelHeurTime"] = 1000
solver.options["TimeLimit"] = 600

res2 = solver.solve(m1, tee=True, warmstart=True)
print("Stage 2 termination:", res2.solver.termination_condition)
print("Stage 2 status:", res2.solver.status)


Read LP format model from file C:\Users\asher\AppData\Local\Temp\tmpsvu7vz85.pyomo.lp
Reading time = 1.74 seconds
x1: 140945 rows, 160025 columns, 499370 nonzeros
Read MIP start from file C:\Users\asher\AppData\Local\Temp\tmpnkdehtf0.gurobi.mst
Set parameter NonConvex to value 2
Set parameter MIPFocus to value 1
Set parameter TimeLimit to value 600
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i5-10210U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  600
MIPFocus  1
NonConvex  2

Optimize a model with 140945 rows, 160025 columns and 499370 nonzeros (Min)
Model fingerprint: 0xd4e1b665
Model has 0 linear objective coefficients
Model has 208115 quadratic objective terms
Variable types: 0 continuous, 160025 integer (160025 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [0e+00, 

In [25]:
# Export event assignments to CSV
start_rows = []
for (e, w, d, s) in m1.X:
    if pyo.value(m1.x[e, w, d, s]) > 0.5:
        start_rows.append((e, w, d, s))

event_output = pd.DataFrame(start_rows, columns=["Event ID", "Week", "Day", "StartSlot"])
event_output = event_output.sort_values(["Week", "Day", "StartSlot", "Event ID"]).reset_index(drop=True)

slot_to_time = {s: f"{9 + (s-1):02d}:00" for s in slots}
event_output["StartTime"] = event_output["StartSlot"].map(slot_to_time)

event_info = (
    EMR_toy[["Event ID", "Event Size", "Room type 2", "Online Delivery"]]
    .drop_duplicates(subset=["Event ID"])
    .rename(columns={
        "Room type 2": "Room Type",
        "Online Delivery": "Online Status"
    })
)

event_info["Online Status"] = (
    event_info["Online Status"]
    .fillna("Not online")
    .replace({
        "Online Live": "Online Live",
        "Online Placeholder": "Online Placeholder"
    })
)

event_info["Campus"] = "Kings Buildings"

event_output = event_output.merge(event_info, on="Event ID", how="left")
event_output["Time Assignment"] = (
    "Week " + event_output["Week"].astype(str)
    + ", " + event_output["Day"].astype(str)
    + ", " + event_output["StartTime"].astype(str)
)

event_output_csv = event_output[
    ["Event ID", "Time Assignment", "Event Size", "Room Type", "Online Status", "Campus"]
]
display(event_output_csv.head(20))

event_output_csv.to_csv("KB_math_60_ShortFri.csv", index=False)
print("Saved: KB_math_60_ShortFri.csv")


,Event ID,Time Assignment,Event Size,Room Type,Online Status,Campus
0,E:3TBAJE9RVV-00005,"Week 26, Friday, 09:00",15.0,General Teaching,Not online,Kings Buildings
1,E:7DVTPHFHXH,"Week 26, Friday, 09:00",12.0,General Teaching,Not online,Kings Buildings
2,E:7DVTPHFHXH-00001,"Week 26, Friday, 09:00",12.0,General Teaching,Not online,Kings Buildings
3,E:8FNL1XQ3FG,"Week 26, Friday, 09:00",45.0,General Teaching,Not online,Kings Buildings
4,E:8WOWA13IS4,"Week 26, Friday, 09:00",40.0,General Teaching,Not online,Kings Buildings
5,E:CR7Z7U6FZ4-00001,"Week 26, Friday, 09:00",12.0,General Teaching,Not online,Kings Buildings
6,E:CR7Z7U6FZ4-00004,"Week 26, Friday, 09:00",12.0,General Teaching,Not online,Kings Buildings
7,E:H5B2GYJZY1,"Week 26, Friday, 09:00",70.0,General Teaching,Not online,Kings Buildings
8,E:QGA6XVHWBB-00008,"Week 26, Friday, 09:00",12.0,General Teaching,Not online,Kings Buildings
9,E:SW37OVIT64-00001,"Week 26, Friday, 09:00",15.0,General Teaching,Not online,Kings Buildings


Saved: KB_math_60_ShortFri.csv


### Add if it online or not `

### Room Assignment

In [26]:
schedule_csv = "KB_math_60_ShortFri.csv"
slot_minutes = 60


In [27]:
### Room Assignment Input

assign_df = pd.read_csv(schedule_csv)

# Keep only in-person events that need a room
assign_df = assign_df[
    (assign_df["Online Status"] == "Not online") &
    assign_df["Room Type"].notna() &
    (assign_df["Room Type"] != "No room required")
].copy()

# Parse "Week 9, Friday, 09:00"
parts = assign_df["Time Assignment"].str.extract(r"Week (\d+), ([^,]+), (\d{2}):(\d{2})")
assign_df["Week"] = parts[0].astype(int)
assign_df["Day"] = parts[1]
assign_df["Hour"] = parts[2].astype(int)
assign_df["Minute"] = parts[3].astype(int)

if slot_minutes == 30:
    assign_df["StartSlot"] = ((assign_df["Hour"] - 9) * 2 + (assign_df["Minute"] // 30) + 1).astype(int)
else:
    assign_df["StartSlot"] = ((assign_df["Hour"] - 9) + 1).astype(int)

# Bring in event durations
event_durations = (
    EMR[["Event ID", "Duration (minutes)"]]
    .dropna(subset=["Event ID", "Duration (minutes)"])
    .drop_duplicates(subset=["Event ID"])
)

assign_df = assign_df.merge(event_durations, on="Event ID", how="left")
assign_df["DurSlots"] = assign_df["Duration (minutes)"].apply(lambda x: int(math.ceil(x / slot_minutes)))

# Unique scheduled occurrence key
assign_df["OccKey"] = list(zip(
    assign_df["Event ID"],
    assign_df["Week"],
    assign_df["Day"],
    assign_df["StartSlot"]
))

display(assign_df.head())
print("Events to room-assign:", len(assign_df))


,Event ID,Time Assignment,Event Size,Room Type,Online Status,Campus,Week,Day,Hour,Minute,StartSlot,Duration (minutes),DurSlots,OccKey
0,E:3TBAJE9RVV-00005,"Week 26, Friday, 09:00",15.0,General Teaching,Not online,Kings Buildings,26,Friday,9,0,1,50,1,"(E:3TBAJE9RVV-00005, 26, Friday, 1)"
1,E:7DVTPHFHXH,"Week 26, Friday, 09:00",12.0,General Teaching,Not online,Kings Buildings,26,Friday,9,0,1,90,2,"(E:7DVTPHFHXH, 26, Friday, 1)"
2,E:7DVTPHFHXH-00001,"Week 26, Friday, 09:00",12.0,General Teaching,Not online,Kings Buildings,26,Friday,9,0,1,90,2,"(E:7DVTPHFHXH-00001, 26, Friday, 1)"
3,E:8FNL1XQ3FG,"Week 26, Friday, 09:00",45.0,General Teaching,Not online,Kings Buildings,26,Friday,9,0,1,50,1,"(E:8FNL1XQ3FG, 26, Friday, 1)"
4,E:8WOWA13IS4,"Week 26, Friday, 09:00",40.0,General Teaching,Not online,Kings Buildings,26,Friday,9,0,1,50,1,"(E:8WOWA13IS4, 26, Friday, 1)"


Events to room-assign: 4248


In [28]:
### Room Candidates and Penalties

rooms_assign = rooms_toy.copy()

def compatible_room(required_type, room_type):
    if pd.isna(required_type) or pd.isna(room_type):
        return False
    if required_type == "General Teaching":
        return True
    if required_type == "Teaching Studio":
        return room_type in {"Teaching Studio", "General Teaching"}
    return room_type == required_type

occurrences = assign_df["OccKey"].tolist()
rooms_list = rooms_assign["Id"].tolist()

occ_req_type = assign_df.set_index("OccKey")["Room Type"].to_dict()
occ_size = assign_df.set_index("OccKey")["Event Size"].to_dict()
occ_week = assign_df.set_index("OccKey")["Week"].to_dict()
occ_day = assign_df.set_index("OccKey")["Day"].to_dict()
occ_start = assign_df.set_index("OccKey")["StartSlot"].to_dict()
occ_len = assign_df.set_index("OccKey")["DurSlots"].to_dict()
occ_event = assign_df.set_index("OccKey")["Event ID"].to_dict()

room_type = rooms_assign.set_index("Id")["Specialist room type"].to_dict()
room_cap = rooms_assign.set_index("Id")["Capacity"].to_dict()

candidate_pairs = []
linear_penalty = {}
quadratic_penalty = {}

for occ in occurrences:
    req = occ_req_type[occ]
    size = float(occ_size[occ])

    for r in rooms_list:
        if compatible_room(req, room_type[r]):
            cap = room_cap[r]
            shortfall = 0 if pd.isna(cap) else max(0.0, size - float(cap))

            candidate_pairs.append((occ, r))
            linear_penalty[(occ, r)] = shortfall
            quadratic_penalty[(occ, r)] = shortfall ** 2

print("Candidate occurrence-room pairs:", len(candidate_pairs))


Candidate occurrence-room pairs: 148605


In [29]:
occupancy = defaultdict(list)

for occ in occurrences:
    w = occ_week[occ]
    d = occ_day[occ]
    s0 = occ_start[occ]
    L = occ_len[occ]

    for t in range(s0, s0 + L):
        occupancy[(w, d, t)].append(occ)

occ_index = list(occupancy.keys())
print("Occupied time indices:", len(occ_index))


Occupied time indices: 350


In [30]:
m2 = pyo.ConcreteModel()

m2.O = pyo.Set(initialize=occurrences, dimen=4)
m2.R = pyo.Set(initialize=rooms_list)
m2.A = pyo.Set(dimen=5, initialize=[(e, w, d, s, r) for ((e, w, d, s), r) in candidate_pairs])
m2.Occ = pyo.Set(dimen=3, initialize=occ_index)

m2.z = pyo.Var(m2.A, domain=pyo.Binary)

# Each occurrence gets exactly one room
def one_room_rule(m, e, w, d, s):
    return sum(m.z[e, w, d, s, r] for r in m.R if (e, w, d, s, r) in m.A) == 1
m2.one_room = pyo.Constraint(m2.O, rule=one_room_rule)

# No room can host two occurrences in the same occupied slot
def no_clash_rule(m, w, d, t, r):
    occs = occupancy[(w, d, t)]
    feasible = [(e, ww, dd, ss, r) for (e, ww, dd, ss) in occs if (e, ww, dd, ss, r) in m.A]
    if not feasible:
        return pyo.Constraint.Skip
    return sum(m.z[e, ww, dd, ss, r] for (e, ww, dd, ss, r) in feasible) <= 1
m2.no_clash = pyo.Constraint(m2.Occ, m2.R, rule=no_clash_rule)


In [31]:
m2.obj = pyo.Objective(
    expr=sum(quadratic_penalty[((e, w, d, s), r)] * m2.z[e, w, d, s, r] for (e, w, d, s, r) in m2.A),
    sense=pyo.minimize
)


In [32]:
solver = pyo.SolverFactory("gurobi")

#solver.options["NonConvex"] = 2
#solver.options["MIPFocus"] = 1
#solver.options["Heuristics"] = 0.4
#solver.options["NoRelHeurTime"] = 1000
solver.options["TimeLimit"] = 100

res_room = solver.solve(m2, tee=True)

print("Room assignment termination:", res_room.solver.termination_condition)
print("Room assignment status:", res_room.solver.status)

linear_total = sum(
    linear_penalty[((e, w, d, s), r)] * pyo.value(m2.z[e, w, d, s, r])
    for (e, w, d, s, r) in m2.A
)

quadratic_total = sum(
    quadratic_penalty[((e, w, d, s), r)] * pyo.value(m2.z[e, w, d, s, r])
    for (e, w, d, s, r) in m2.A
)

print("Linear penalty:", linear_total)
print("Quadratic penalty:", quadratic_total)


Read LP format model from file C:\Users\asher\AppData\Local\Temp\tmpblgr4i0u.pyomo.lp
Reading time = 1.05 seconds
x1: 16848 rows, 148605 columns, 360786 nonzeros
Set parameter TimeLimit to value 100
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i5-10210U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  100

Optimize a model with 16848 rows, 148605 columns and 360786 nonzeros (Min)
Model fingerprint: 0x6a75ecbf
Model has 35001 linear objective coefficients
Variable types: 0 continuous, 148605 integer (148605 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 2e+05]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Found heuristic solution: objective 1.643089e+07
Presolve removed 10905 rows and 91728 columns (presolve time = 5s)...
Presolve removed 16444 

In [33]:
assigned_rooms = []
for (e, w, d, s, r) in m2.A:
    if pyo.value(m2.z[e, w, d, s, r]) > 0.5:
        assigned_rooms.append((e, w, d, s, r, room_cap[r], room_type[r]))

assigned_rooms_df = pd.DataFrame(
    assigned_rooms,
    columns=["Event ID", "Week", "Day", "StartSlot", "Assigned Room", "Room Capacity", "Assigned Room Type"]
)

final_room_output = assign_df.merge(
    assigned_rooms_df,
    on=["Event ID", "Week", "Day", "StartSlot"],
    how="left"
)

display(final_room_output.head(20))

final_room_output.to_csv("KB_math_60_ShortFri_complete_assignments.csv", index=False)
print("Saved: KB_math_60_ShortFri_complete_assignments.csv")


,Event ID,Time Assignment,Event Size,Room Type,Online Status,Campus,Week,Day,Hour,Minute,StartSlot,Duration (minutes),DurSlots,OccKey,Assigned Room,Room Capacity,Assigned Room Type
0,E:3TBAJE9RVV-00005,"Week 26, Friday, 09:00",15.0,General Teaching,Not online,Kings Buildings,26,Friday,9,0,1,50,1,"(E:3TBAJE9RVV-00005, 26, Friday, 1)",0613_01_1.1104,104.0,Laboratory
1,E:7DVTPHFHXH,"Week 26, Friday, 09:00",12.0,General Teaching,Not online,Kings Buildings,26,Friday,9,0,1,90,2,"(E:7DVTPHFHXH, 26, Friday, 1)",0613_08_8.8216,50.0,Laboratory
2,E:7DVTPHFHXH-00001,"Week 26, Friday, 09:00",12.0,General Teaching,Not online,Kings Buildings,26,Friday,9,0,1,90,2,"(E:7DVTPHFHXH-00001, 26, Friday, 1)",0613_04_4.4325C,48.0,General Teaching
3,E:8FNL1XQ3FG,"Week 26, Friday, 09:00",45.0,General Teaching,Not online,Kings Buildings,26,Friday,9,0,1,50,1,"(E:8FNL1XQ3FG, 26, Friday, 1)",0613_01_1.1208,66.0,Computer Laboratory
4,E:8WOWA13IS4,"Week 26, Friday, 09:00",40.0,General Teaching,Not online,Kings Buildings,26,Friday,9,0,1,50,1,"(E:8WOWA13IS4, 26, Friday, 1)",0613_02_2.3905,150.0,General Teaching
5,E:CR7Z7U6FZ4-00001,"Week 26, Friday, 09:00",12.0,General Teaching,Not online,Kings Buildings,26,Friday,9,0,1,80,2,"(E:CR7Z7U6FZ4-00001, 26, Friday, 1)",0613_02_2.2901,14.0,General Teaching
6,E:CR7Z7U6FZ4-00004,"Week 26, Friday, 09:00",12.0,General Teaching,Not online,Kings Buildings,26,Friday,9,0,1,80,2,"(E:CR7Z7U6FZ4-00004, 26, Friday, 1)",0613_02_2.2205,12.0,Laboratory
7,E:H5B2GYJZY1,"Week 26, Friday, 09:00",70.0,General Teaching,Not online,Kings Buildings,26,Friday,9,0,1,110,2,"(E:H5B2GYJZY1, 26, Friday, 1)",0613_06_6.6231,90.0,Laboratory
8,E:QGA6XVHWBB-00008,"Week 26, Friday, 09:00",12.0,General Teaching,Not online,Kings Buildings,26,Friday,9,0,1,90,2,"(E:QGA6XVHWBB-00008, 26, Friday, 1)",0613_05_5.5327,50.0,General Teaching
9,E:SW37OVIT64-00001,"Week 26, Friday, 09:00",15.0,General Teaching,Not online,Kings Buildings,26,Friday,9,0,1,50,1,"(E:SW37OVIT64-00001, 26, Friday, 1)",0613_04_4.4205,15.0,Laboratory


Saved: KB_math_60_ShortFri_complete_assignments.csv
